In [33]:
import numpy as np
import pandas as pd
import tensorflow as tf

import keras
from keras import layers

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score

In [34]:
IMG_DIRECTORY = "data/images"
ARCH = "convnext" # to select the model to use, options: convnext, resnet50, efficientnet
IMAGE_SIZE = 224
BATCH = 64
EPOCHS_HEAD = 5 # for feature extraction purposes
EPOCHS_FINE_TUNING = 20 # to fine tune
LR_HEAD = 1e-3
LR_FINE = 3e-5
SEED = 42

In [35]:
keras.utils.set_random_seed(SEED)
train = pd.read_pickle("train.pkl")
validation = pd.read_pickle("validation.pkl")
test = pd.read_pickle("test.pkl")

In [36]:
for image in (train, validation, test):
    image['path'] = IMG_DIRECTORY + "/" + image['filename']

In [37]:
NUM_CLASSES = train['encoded_id'].nunique()
print(len(train), len(validation), len(test))

41991 8998 8999


Since the dataset is imbalanced:

In [38]:
class_weights = compute_class_weight("balanced", classes = np.arange(NUM_CLASSES), y = train['encoded_id'].values)

class_weights = tf.constant(class_weights, dtype = tf.float32)

In [39]:
def load_image(path):
    
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels = 3) # since we have RBG images, there are 3 channels
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE), method = 'bicubic')
    return tf.clip_by_value(image, 0.0, 255.0)

In [40]:
def make_tf_dataset(dataframe, training):

    paths = dataframe['path'].values
    ids = dataframe['encoded_id'].values.astype('int32')

    dataset = tf.data.Dataset.from_tensor_slices((paths, ids))

    if training:
        dataset = dataset.shuffle(len(dataframe), seed = SEED, reshuffle_each_iteration = True)

    def _map(path, id): # the _ symbolises an internal function
        x = load_image(path)
        
        if training:
            return x, id, tf.gather(class_weights, id)
        return x, id
    
    return dataset.map(_map, num_parallel_calls = tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [41]:
train_ds = make_tf_dataset(train, True)
val_ds = make_tf_dataset(validation, False)
test_ds = make_tf_dataset(test, False)

In [42]:
def build_model():
    shape = (IMAGE_SIZE, IMAGE_SIZE, 3)

    if ARCH == "convnext":

        base = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet", input_shape=shape)
        prep = layers.Identity()

    elif ARCH == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=shape)
        prep = layers.Lambda(keras.applications.resnet50.preprocess_input)

    elif ARCH == "efficientnet":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=shape)
        prep = layers.Identity()

    else:
        raise ValueError(ARCH)

    input = keras.Input(shape)
    x = prep(input)
    x = base(x, training = False)

    x = layers.GlobalAveragePooling2D()(x)
    #x = layers.Dropout(0.3)(x)

    output = layers.Dense(NUM_CLASSES, activation = 'softmax')(x)

    return keras.Model(input, output), base

In [43]:
model, base = build_model()

loss_function = tf.keras.losses.SparseCategoricalCrossentropy()

metrics = ["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name = "top5")]

In [44]:
base.trainable = False

model.compile(optimizer = keras.optimizers.AdamW(LR_HEAD), loss = loss_function, metrics = metrics)

model.fit(train_ds, validation_data = val_ds, epochs = EPOCHS_HEAD)

Epoch 1/5


/opt/anaconda3/envs/deep_learning/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1794078892.963511  197388 service.cc:153] XLA service 0x706294dc50 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1794078892.963599  197388 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1794078892.983787  197388 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 59/657 ━━━━━━━━━━━━━━━━━━━━ 1:06:45 7s/step - accuracy: 0.1014 - loss: 5.1171 - top5: 0.1986

KeyboardInterrupt: 

In [45]:
base.trainable = True
steps = EPOCHS_FINE_TUNING * len(train_ds)

lr = keras.optimizers.schedules.CosineDecay(LR_FINE, decay_steps=steps)

model.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay = 1e-4), loss = loss_function, metrics = metrics)

callbacks = [
    keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 4, restore_best_weights = True),
    keras.callbacks.ModelCheckpoint(f"best_{ARCH}.keras", monitor = "val_loss", save_best_only = True),
]
model.fit(train_ds, validation_data = val_ds, epochs = EPOCHS_FINE_TUNING, callbacks = callbacks)

Epoch 1/20
  2/657 ━━━━━━━━━━━━━━━━━━━━ 4:33:03 25s/step - accuracy: 0.2891 - loss: 4.3512 - top5: 0.2112 

KeyboardInterrupt: 

In [ ]:
y_true = test["encoded_id"].values

y_pred = model.predict(test_ds).argmax(axis = 1)

print("Test accuracy:", (y_true == y_pred).mean())
print("Test macro-F1:", f1_score(y_true, y_pred, average = "macro"))

names = train.drop_duplicates("endoded_id").set_index("encoded_id")["description"]

print(classification_report(y_true, y_pred, target_names=[names[i][:25] for i in range(NUM_CLASSES)]))